## Training

Please note that the data we trained on cannot be publically shared.


In [ ]:
# Install required packages
!pip install numpy scikit-learn torch -q
!pip install rdkit xgboost -q
!pip install pandas

!pip install torch
!pip install transformers
!pip install sentencepiece

!pip install chemprop -q

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [ ]:
import pandas as pd
import numpy as np

train_data = pd.read_csv('train.csv')


In [ ]:
#Import models
import numpy as np
import torch
from transformers import T5Tokenizer, T5EncoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc')
model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_half_uniref50-enc",
    torch_dtype=torch.float16
)
model = model.to(device).eval()


In [ ]:
#Compute the embedding given a certain sequence
protein_cache = {}
def get_protein_embedding(sequences, batch_size = 2):
  uncached = [seq for seq in sequences if seq not in protein_cache]

  for start in range(0, len(uncached), batch_size):
    batch = uncached[start:start + batch_size]

    batch_spaced = [" ".join(list(seq)) for seq in batch]

    inputs = tokenizer(batch_spaced, return_tensors="pt", padding = True, truncation = True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    residue_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_embeddings = residue_embeddings * attention_mask
    sum_embeddings = masked_embeddings.sum(dim=1)

    batch_embeddings = (sum_embeddings / attention_mask.sum(dim=1)).cpu().numpy()

    for seq, emb in zip(batch, batch_embeddings):
      protein_cache[seq] = emb


In [ ]:
# From the data, find the number of valid (i.e. non-NULL proteins) and get list of unique ones.
df       = train_data.copy()
protein  = df["amino_acid_sequence"].values
drugs    = df["SMILES"].values
affinity = df["Affinity"].values
valid_protein = [seq for seq in protein if isinstance(seq, str) and len(seq) > 0]
unique_proteins = list(set(valid_protein))

# Fix index
valid_mask = np.array([isinstance(seq, str) and len(seq) > 0 for seq in protein])
df         = df[valid_mask].reset_index(drop=True)
protein    = df["amino_acid_sequence"].values
drugs      = df["SMILES"].values
affinity   = df["Affinity"].values

unique_proteins = list(set(protein))
print(f"Unique proteins: {len(unique_proteins)}, Total rows: {len(protein)}")

# Get embedding of all unique proteins from ProtTrans
get_protein_embedding(unique_proteins, batch_size=16)

seq_to_emb = {seq: protein_cache[seq] for seq in unique_proteins}

train_data["protein_emb"] = train_data["amino_acid_sequence"].map(seq_to_emb)

# Map our proteins to their embedding based on the seq_to_emb dictionary
np.save("protein_embeddings.npy",
        np.array(list(seq_to_emb.values())).astype(np.float32))
print(f"Unique proteins embedded: {len(seq_to_emb)}")

In [ ]:
# Delete model to clear space
del model
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
train_data.head()

In [ ]:

# Save embeddings to skip future computation
from google.colab import drive
drive.mount('/content/drive')

import pickle
with open("/content/drive/MyDrive/seq_to_emb.pkl", "wb") as f:
    pickle.dump(seq_to_emb, f)

train_data.to_pickle("/content/drive/MyDrive/train_with_embeddings.pkl")
print(f"Saved {len(seq_to_emb)} protein embeddings to Drive!")

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from rdkit.Chem import BRICS
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator


# Create fingerprint generator, looks two mol away
fp_gen = GetMorganGenerator(radius=2, fpSize=2048)

def get_fragment_embedding(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(2048)
    # Fragment the molecule
    frags = list(BRICS.BRICSDecompose(mol))
    if not frags:
        return np.array(fp_gen.GetFingerprint(mol)).astype(np.float32)
    fps = []
    for f in frags:
        fmol = Chem.MolFromSmiles(f)
        if fmol is not None:
            fps.append(np.array(fp_gen.GetFingerprint(fmol)).astype(np.float32))
    # Returns the average of the fragment fingerprints
    return np.mean(fps, axis=0) if fps else np.zeros(2048)

# cache to avoid recomputing same SMILES
frag_cache = {}
def get_fragment_embedding_cached(smiles):
    if smiles in frag_cache:
        return frag_cache[smiles]
    result = get_fragment_embedding(smiles)
    frag_cache[smiles] = result
    return result

X_mols = np.stack([get_fragment_embedding_cached(s) for s in train_data["SMILES"]])
print(f"Fragment embeddings shape: {X_mols.shape}")



In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from rdkit.Chem import BRICS
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

In [ ]:
# This cell is where you run from if you have the clean data files already

import pandas as pd
import numpy as np
import pickle

# load train.csv from Colab environment
train_data = pd.read_csv('train.csv')
train_data = train_data.dropna(subset=["amino_acid_sequence"]).reset_index(drop=True)
print(f"Loaded {len(train_data)} rows")

# load saved files from Drive
with open("/content/drive/MyDrive/seq_to_emb.pkl", "rb") as f:
    seq_to_emb = pickle.load(f)

with open("/content/drive/MyDrive/frag_cache.pkl", "rb") as f:
    frag_cache = pickle.load(f)


In [ ]:
t

In [ ]:
X_proteins = np.array([seq_to_emb[seq] for seq in train_data["amino_acid_sequence"]]).astype(np.float32)
targets_np  = np.array(train_data["Affinity"]).astype(np.float32)

# Combine drug and target embeds
X_combined = np.hstack([X_mols, X_proteins])
y = targets_np
print(f"Combined shape: {X_combined.shape}")

# protein-based split
proteins    = train_data["amino_acid_sequence"].values
unique_prots = list(set(proteins))

train_prot, temp_prot = train_test_split(unique_prots, test_size=0.30, random_state=42)
val_prot,   test_prot = train_test_split(temp_prot,    test_size=0.50, random_state=42)

train_mask = np.array([p in set(train_prot) for p in proteins])
val_mask   = np.array([p in set(val_prot)   for p in proteins])
test_mask  = np.array([p in set(test_prot)  for p in proteins])

X_reg_train, y_reg_train = X_combined[train_mask], y[train_mask]
X_reg_valid, y_reg_valid = X_combined[val_mask],   y[val_mask]
X_reg_test,  y_reg_test  = X_combined[test_mask],  y[test_mask]

print(f"Train: {len(X_reg_train)}, Val: {len(X_reg_valid)}, Test: {len(X_reg_test)}")

# models — dropped SVR and KNN (too slow on 331K rows)
reg_models = {
    "Lasso":             make_pipeline(StandardScaler(), Lasso(alpha=0.1)),
    "XGBoost":           XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.03,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                             random_state=42, verbosity=0, tree_method="hist", device="cuda"),
}



In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_regression(y_true, y_pred):
    return {
        'MSE': mean_squared_error(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2':  r2_score(y_true, y_pred),
    }

reg_models = {
    "Lasso":   make_pipeline(StandardScaler(), Lasso(alpha=0.1)),
    "XGBoost": XGBRegressor(
    n_estimators=5000,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.8,
    objective="reg:squarederror",
    colsample_bytree=0.7,
    tree_method="hist",
    early_stopping_rounds=50,
    reg_lambda=1.0,
    random_state=42,
    device="cuda"
)
,
}

reg_results = {}
for name, model in reg_models.items():
    print(f"Training {name}...")
    if name == "XGBoost":
        model.fit(X_reg_train, y_reg_train,
                  eval_set=[(X_reg_valid, y_reg_valid)],
                  verbose=100)
    else:
        model.fit(X_reg_train, y_reg_train)
    reg_results[name] = {
        "model":       model,
        "train_preds": model.predict(X_reg_train),
        "valid_preds": model.predict(X_reg_valid),
        "test_preds":  model.predict(X_reg_test),
    }
    metrics = evaluate_regression(y_reg_valid, reg_results[name]["valid_preds"])
    print(f"  Valid R²: {metrics['R2']:.4f} | MSE: {metrics['MSE']:.4f} | MAE: {metrics['MAE']:.4f}")

print(f"\n{'Model':<22} {'Split':<8} {'MSE':>8} {'MAE':>8} {'R2':>8}")
print("-" * 58)
for name, res in reg_results.items():
    for split_name, y_true, y_pred in [
        ('Train', y_reg_train, res['train_preds']),
        ('Valid', y_reg_valid, res['valid_preds']),
        ('Test',  y_reg_test,  res['test_preds']),
    ]:
        metrics = evaluate_regression(y_true, y_pred)
        print(f"{name:<22} {split_name:<8} {metrics['MSE']:>8.2f} {metrics['MAE']:>8.2f} {metrics['R2']:>8.3f}")

In [ ]:
import pickle
from google.colab import drive
drive.mount('/content/drive')

# save XGBoost (your best model)
with open("/content/drive/MyDrive/xgb_final.pkl", "wb") as f:
    pickle.dump(reg_results["XGBoost"]["model"], f)

print("Model saved!")

## Code for Testing on Dataset

In [ ]:
import pickle
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from rdkit.Chem import BRICS
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from transformers import T5Tokenizer, T5EncoderModel
from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/MyDrive/xgb_final.pkl", "rb") as f:
    brics_model = pickle.load(f)
with open("/content/drive/MyDrive/seq_to_emb.pkl", "rb") as f:
    seq_to_emb = pickle.load(f)
with open("/content/drive/MyDrive/frag_cache.pkl", "rb") as f:
    frag_cache = pickle.load(f)

# load ProtT5 for unseen proteins
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc')
prot_model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_half_uniref50-enc", torch_dtype=torch.float16
).to(device).eval()

def embed_sequence(seq):
    if seq in seq_to_emb:
        return seq_to_emb[seq]
    inputs = tokenizer([" ".join(list(seq))], return_tensors="pt",
                       padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = prot_model(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1)
    emb  = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
    result = emb.cpu().float().numpy()[0]
    seq_to_emb[seq] = result
    return result

fp_gen = GetMorganGenerator(radius=2, fpSize=2048)

def get_fragment_embedding(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(2048)
    frags = list(BRICS.BRICSDecompose(mol))
    if not frags:
        return np.array(fp_gen.GetFingerprint(mol)).astype(np.float32)
    fps = []
    for f in frags:
        fmol = Chem.MolFromSmiles(f)
        if fmol is not None:
            fps.append(np.array(fp_gen.GetFingerprint(fmol)).astype(np.float32))
    return np.mean(fps, axis=0) if fps else np.zeros(2048)

def get_fragment_embedding_cached(smiles):
    if smiles in frag_cache:
        return frag_cache[smiles]
    result = get_fragment_embedding(smiles)
    frag_cache[smiles] = result
    return result

test_warm         = pd.read_csv('test_full_warm.csv')
test_cold_drug    = pd.read_csv('test_cold_drug.csv')
test_cold_protein = pd.read_csv('test_cold_target.csv')
test_cold_full    = pd.read_csv('test_full_cold.csv')

def featurize_test(df):
    X_mols     = np.stack([get_fragment_embedding_cached(s) for s in tqdm(df["SMILES"], desc="SMILES")])
    X_proteins = np.array([embed_sequence(seq) for seq in tqdm(df["amino_acid_sequence"], desc="Proteins")]).astype(np.float32)
    return np.hstack([X_mols, X_proteins])

print(f"\n{'Split':<15} | {'R²':>8} | {'MSE':>8} | {'MAE':>8}")
print("-" * 48)
for name, df in [
    ("Warm",         test_warm),
    ("Cold Drug",    test_cold_drug),
    ("Cold Protein", test_cold_protein),
    ("Cold Full",    test_cold_full),
]:
    X     = featurize_test(df)
    preds = brics_model.predict(X)
    r2    = r2_score(df["Affinity"], preds)
    mse   = mean_squared_error(df["Affinity"], preds)
    mae   = mean_absolute_error(df["Affinity"], preds)
    print(f"{name:<15} | {r2:>8.4f} | {mse:>8.4f} | {mae:>8.4f}")

## Legacy Code (MLP)




In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import TensorDataset, DataLoader
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# def evaluate_regression(y_true, y_pred):
#     return {
#         'MSE': mean_squared_error(y_true, y_pred),
#         'MAE': mean_absolute_error(y_true, y_pred),
#         'R2':  r2_score(y_true, y_pred),
#     }

# # --- tensors ---
# X_train_t = torch.tensor(X_reg_train, dtype=torch.float32).to(device)
# X_valid_t = torch.tensor(X_reg_valid, dtype=torch.float32).to(device)
# y_train_t = torch.tensor(y_reg_train, dtype=torch.float32).to(device)

# # --- dataloaders ---
# train_ds     = TensorDataset(X_train_t, y_train_t)
# train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

# # --- MLP ---
# class MLP(nn.Module):
#     def __init__(self, input_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 512),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )
#     def forward(self, x):
#         return self.net(x).squeeze(1)

# mlp       = MLP(input_dim=X_train_t.shape[1]).to(device)
# optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
# loss_fn   = nn.MSELoss()
# best_r2   = -float('inf')

# for epoch in range(100):
#     mlp.train()
#     for X_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#         loss = loss_fn(mlp(X_batch), y_batch)
#         loss.backward()
#         optimizer.step()

#     if (epoch + 1) % 10 == 0:
#         mlp.eval()
#         with torch.no_grad():
#             val_preds = mlp(X_valid_t).cpu().numpy()
#         metrics = evaluate_regression(y_reg_valid, val_preds)
#         if metrics['R2'] > best_r2:
#             best_r2 = metrics['R2']
#             torch.save(mlp.state_dict(), "best_mlp.pt")
#         print(f"Epoch {epoch+1:3d} | R²: {metrics['R2']:.4f} | MSE: {metrics['MSE']:.4f} | MAE: {metrics['MAE']:.4f}")

# print(f"\nBest MLP R²: {best_r2:.4f}")

# # --- sklearn results ---
# print(f"\n{'Model':<22} {'Split':<8} {'MSE':>8} {'MAE':>8} {'R2':>8}")
# print("-" * 58)
# for name, res in reg_results.items():
#     metrics = evaluate_regression(y_reg_valid, res['valid_preds'])
#     print(f"{name:<22} {'Valid':<8} {metrics['MSE']:>8.2f} {metrics['MAE']:>8.2f} {metrics['R2']:>8.3f}")

In [ ]:
# import torch
# import torch.nn as nn
# from chemprop.data import build_dataloader, MoleculeDataset, MoleculeDatapoint
# from chemprop.nn import BondMessagePassing, MeanAggregation
# from rdkit import Chem

# # filter train_data to only valid SMILES before splitting
# valid_mask  = train_data["SMILES"].apply(lambda s: Chem.MolFromSmiles(s) is not None)
# train_data  = train_data[valid_mask].reset_index(drop=True)
# X_proteins  = X_proteins[valid_mask.values]
# targets_np  = targets_np[valid_mask.values]

# print(f"Valid rows after filtering: {len(train_data)}")
# # split indices
# n         = len(targets_np)
# indices   = np.random.permutation(n)
# split     = int(0.9 * n)
# train_idx = indices[:split]
# val_idx   = indices[split:]

# train_smiles = train_data["SMILES"].iloc[train_idx].tolist()
# val_smiles   = train_data["SMILES"].iloc[val_idx].tolist()

# dataset_train = MoleculeDataset([MoleculeDatapoint.from_smi(s) for s in train_smiles])
# dataset_val   = MoleculeDataset([MoleculeDatapoint.from_smi(s) for s in val_smiles])

# X_prot_train = torch.tensor(X_proteins[train_idx], dtype=torch.float32).to(device)
# X_prot_val   = torch.tensor(X_proteins[val_idx],   dtype=torch.float32).to(device)
# y_train      = torch.tensor(targets_np[train_idx],  dtype=torch.float32).to(device)
# y_val        = torch.tensor(targets_np[val_idx],    dtype=torch.float32).to(device)

# train_loader = build_dataloader(dataset_train, batch_size=64, shuffle=False)
# val_loader   = build_dataloader(dataset_val,   batch_size=64, shuffle=False)

# class DTIModel(nn.Module):
#     def __init__(self, mol_dim, prot_dim):
#         super().__init__()
#         self.mp  = BondMessagePassing()
#         self.agg = MeanAggregation()
#         self.mlp = nn.Sequential(
#             nn.Linear(mol_dim + prot_dim, 512),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )

#     def forward(self, bmg, prot_emb):
#         bmg.V          = bmg.V.to(device)
#         bmg.E          = bmg.E.to(device)
#         bmg.edge_index = bmg.edge_index.to(device)
#         bmg.batch      = bmg.batch.to(device)
#         h   = self.mp(bmg)
#         mol = self.agg(h, bmg.batch)
#         return self.mlp(torch.cat([mol, prot_emb], dim=1)).squeeze(1)

# dti_model = DTIModel(mol_dim=300, prot_dim=1024).to(device)
# optimizer  = torch.optim.Adam(dti_model.parameters(), lr=1e-3)
# loss_fn    = nn.MSELoss()
# best_r2    = -float('inf')

# for epoch in range(50):
#     dti_model.train()
#     for i, batch in enumerate(train_loader):
#         bmg      = batch.bmg
#         prot_emb = X_prot_train[i*64:(i+1)*64]
#         y_batch  = y_train[i*64:(i+1)*64]
#         optimizer.zero_grad()
#         loss = loss_fn(dti_model(bmg, prot_emb), y_batch)
#         loss.backward()
#         optimizer.step()

#     dti_model.eval()
#     preds_all, targets_all = [], []
#     with torch.no_grad():
#         for i, batch in enumerate(val_loader):
#             preds_all.append(dti_model(batch.bmg, X_prot_val[i*64:(i+1)*64]).cpu().numpy())
#             targets_all.append(targets_np[val_idx][i*64:(i+1)*64])

#     preds_all   = np.concatenate(preds_all)
#     targets_all = np.concatenate(targets_all)
#     metrics     = evaluate_regression(targets_all, preds_all)

#     if metrics['R2'] > best_r2:
#         best_r2 = metrics['R2']
#         torch.save(dti_model.state_dict(), "best_dti.pt")

#     if (epoch + 1) % 10 == 0:
#         print(f"Epoch {epoch+1:3d} | R²: {metrics['R2']:.4f} | MSE: {metrics['MSE']:.4f} | MAE: {metrics['MAE']:.4f}")

# print(f"\nBest DTI R²: {best_r2:.4f}")